# 09_misclass_compare

원본 Colab 셀에서 분리. (`#@title 분류오류 대조: 정답 알약 vs 모델이 답한 알약`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [매 세션 3] dataset_74 zip 복원
import os                                                     # 경로 확인 도구

ZIP = os.path.join(PROJ_ROOT, "dataset_74_5fold.zip")         # 74클래스 5-fold zip(드라이브)
print("zip 존재:", os.path.exists(ZIP))                        # True 확인

!cp "$ZIP" /content/                                           # 드라이브 → 로컬 복사(속도 위해)
!unzip -qo /content/dataset_74_5fold.zip -d /content/dataset_74  # 압축 해제(-o=덮어쓰기)
print("복원 fold:", sorted(d for d in os.listdir("/content/dataset_74") if d.startswith("fold")))

In [ ]:
#@title 분류오류 대조: 정답 알약 vs 모델이 답한 알약
#@markdown 분류오류 각 건에 대해 [실제 그 자리의 알약 크롭] / [GT 클래스 대표 샘플] / [모델이 답한 클래스 대표 샘플] 3장을 나란히 출력. 모델이 왜 헷갈렸는지, 혹은 GT가 틀렸는지 육안 판단용

import os, json, glob, unicodedata                             # 경로·json·검색·정규화 도구
import pandas as pd                                             # CSV 읽기 도구
import matplotlib.pyplot as plt                                 # 출력 도구
import matplotlib.font_manager as fm                            # 폰트 등록 도구
from PIL import Image                                           # 이미지 크롭 도구
from pathlib import Path                                        # 경로 객체 도구

for p in glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic*.ttf"):
    fm.fontManager.addfont(p)
plt.rc("font", family="NanumGothic")                            # 한글 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

TEAM_ROOT = Path("/content/drive/MyDrive/1팀 공유 문서")
USER_ROOT = TEAM_ROOT / "김태윤"
DATASET = Path("/content/dataset_74")
ERR_CSV = USER_ROOT / "error_analysis_74cls" / "valid_errors_74cls.csv"
LABEL_MAP_PATH = USER_ROOT / "label_map_74.json"

PAD = 0.15                                                       # 크롭 여백 비율
N_SAMPLES = 3                                                    # 클래스 대표 샘플 개수

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    id2kcode = {int(k): int(v) for k, v in json.load(f).items()}
kcode2id = {v: k for k, v in id2kcode.items()}                   # K코드 -> 내부id 역방향

# ===== 전체 fold json을 훑어 (내부id -> 그 클래스 인스턴스 목록) 인덱스 구축 =====
inst_by_cat = {}                                                 # {내부id: [(이미지경로, bbox), ...]}
file_index = {}                                                  # {파일명: (이미지경로, {ann_id: (bbox, cat)})}

for jf in glob.glob(f"{DATASET}/fold*/*/_annotations.coco.json"):
    img_dir = Path(jf).parent
    with open(jf, "r", encoding="utf-8") as f:
        coco = json.load(f)
    id2file = {im["id"]: im["file_name"] for im in coco["images"]}

    for ann in coco["annotations"]:
        fname = id2file[ann["image_id"]]
        fpath = img_dir / fname
        cat = ann["category_id"]
        inst_by_cat.setdefault(cat, []).append((fpath, ann["bbox"]))   # 클래스별 인스턴스 수집

        if fname not in file_index:
            file_index[fname] = (fpath, {})
        file_index[fname][1][ann["id"]] = (ann["bbox"], cat)            # ann_id로 바로 찾기 위함

def crop(fpath, bbox, pad=PAD):
    # bbox 주변을 여백 포함해 크롭하는 도구 #bbox=[x,y,w,h]
    img = Image.open(fpath).convert("RGB")
    x, y, w, h = bbox
    px, py = w * pad, h * pad
    left, top = max(0, int(x - px)), max(0, int(y - py))
    right, bottom = min(img.width, int(x + w + px)), min(img.height, int(y + h + py))
    if right <= left or bottom <= top:                            # 손상 bbox 방어
        return None
    return img.crop((left, top, right, bottom))

# ===== 분류오류 건별 3장 대조 출력 =====
err = pd.read_csv(ERR_CSV)
cls_err = err[err["error_type"] == "분류오류"]
print(f"분류오류 {len(cls_err)}건\n")

for r in cls_err.itertuples():
    gt_id, pred_id = int(r.gt_cat), int(r.pred_cat)
    gt_k, pred_k = int(r.gt_kcode), int(r.pred_kcode)

    fpath, anns = file_index[r.file_name]                          # 문제 이미지·annotation 조회
    gt_bbox, _ = anns[int(r.ann_id)]                                # 이 오류의 실제 GT 박스

    fig, axes = plt.subplots(1, 1 + N_SAMPLES * 2, figsize=(3 * (1 + N_SAMPLES * 2), 3.6))

    # [1] 실제 그 자리의 알약 (GT 박스로 크롭)
    c = crop(fpath, gt_bbox)
    axes[0].imshow(c) if c else None
    axes[0].set_title(f"실제 이 자리 알약\nGT라벨={gt_k}", fontsize=8, color="green")
    axes[0].axis("off")

    # [2] GT 클래스의 다른 대표 샘플들 (이 클래스는 원래 어떻게 생겼나)
    for i, (sp, sb) in enumerate(inst_by_cat[gt_id][:N_SAMPLES]):
        ax = axes[1 + i]
        c = crop(sp, sb)
        ax.imshow(c) if c else None
        ax.set_title(f"GT클래스 {gt_k}\n대표샘플", fontsize=8, color="green")
        ax.axis("off")

    # [3] 모델이 답한 클래스의 대표 샘플들 (모델은 이렇게 생긴 걸로 봤다)
    for i, (sp, sb) in enumerate(inst_by_cat[pred_id][:N_SAMPLES]):
        ax = axes[1 + N_SAMPLES + i]
        c = crop(sp, sb)
        ax.imshow(c) if c else None
        ax.set_title(f"예측클래스 {pred_k}\n대표샘플", fontsize=8, color="red")
        ax.axis("off")

    fig.suptitle(f"{r.file_name} | ann_id={int(r.ann_id)} | IoU={r.iou} score={r.score}", fontsize=9)
    plt.tight_layout()
    plt.show()